<div style="text-align: center;">
<pre style="display: inline-block; text-align: left;">
Fichiers IEEE txt
        ↓
Extraction des articles
        ↓
Pour chaque article :
    titre + abstract + keywords
        ↓
Prompt envoyé à Qwen/Ollama
        ↓
Réponse JSON : include / exclude / uncertain
        ↓
Sauvegarde checkpoint
        ↓
Export final Excel + CSV
</pre>
</div>

In [3]:
import re
import json
import time
import requests
import pandas as pd
from pathlib import Path


start = time.perf_counter()

MODEL = "qwen3:8b"

FILES = [
    "ieee_results_1.txt",
    "ieee_results_2.txt"
]

OUTPUT_EXCEL = "screening_results.xlsx"
OUTPUT_CSV = "screening_results.csv"

CHECKPOINT_CSV = "screening_checkpoint.csv"
CHECKPOINT_EXCEL = "screening_checkpoint.xlsx"

TIMEOUT_SECONDS = 180


INCLUSION_CRITERIA = """
Include only original research papers related to visuo-haptic / visuo-tactile perception.

The paper should involve at least two modalities among:
- vision / visual / RGB / camera
- tactile / touch / haptic / force / pressure / texture sensing

The paper should focus on perception-level tasks such as:
- object recognition
- object classification
- material recognition
- texture recognition or analysis
- surface property recognition
- 3D shape recognition
- object property or attribute recognition
- multimodal or cross-modal representation learning
- visual-tactile or visual-haptic fusion
"""

EXCLUSION_CRITERIA = """
Exclude papers if:
- the main focus is grasping, grasp planning, manipulation, robot control, trajectory planning, or pose estimation
- the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception
- the paper is a review, survey, tutorial, editorial, or non-original study
- the abstract does not clearly involve both visual and tactile/haptic information
- the task is not related to perception or recognition
"""


def split_ieee_txt(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\xa0", " ")

    text = re.sub(r'(?<!\n)(Abstract:)', r'\n\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(?<!\n)(keywords:)', r'\n\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(?<!\n)(URL:)', r'\n\1', text, flags=re.IGNORECASE)

    pattern = (
        r'(.*?URL:\s*https?://ieeexplore\.ieee\.org/stamp/stamp\.jsp'
        r'\?tp=&arnumber=\d+&isnumber=\d+)'
    )

    chunks = re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE)

    papers = []

    print("Detected URLs:", len(re.findall(r"URL:", text, flags=re.IGNORECASE)))
    print("Detected paper chunks:", len(chunks))

    for chunk in chunks:
        chunk = chunk.strip()

        citation_match = re.search(
            r'^(.*?)(?=\nAbstract:)',
            chunk,
            flags=re.DOTALL | re.IGNORECASE
        )
        citation = citation_match.group(1).strip() if citation_match else ""

        title_match = re.search(r'"([^"]+)"', citation)
        title = title_match.group(1).strip() if title_match else ""

        authors = citation.split('"')[0].strip().rstrip(",") if '"' in citation else ""

        abstract_match = re.search(
            r'Abstract:\s*(.*?)(?=\nkeywords:|\nURL:|$)',
            chunk,
            flags=re.DOTALL | re.IGNORECASE
        )
        abstract = abstract_match.group(1).strip() if abstract_match else ""
        abstract = re.sub(r"\s+", " ", abstract)

        keywords_match = re.search(
            r'keywords:\s*\{?(.*?)\}?,?\s*(?=\nURL:|$)',
            chunk,
            flags=re.DOTALL | re.IGNORECASE
        )
        keywords = keywords_match.group(1).strip() if keywords_match else ""
        keywords = re.sub(r"\s+", " ", keywords)

        doi_match = re.search(r"doi:\s*([^\s,]+)", citation, flags=re.IGNORECASE)
        doi = doi_match.group(1).strip().rstrip(".") if doi_match else ""

        url_match = re.search(r"URL:\s*(https?://\S+)", chunk, flags=re.IGNORECASE)
        url = url_match.group(1).strip() if url_match else ""

        papers.append({
            "title": title,
            "authors": authors,
            "abstract": abstract,
            "keywords": keywords,
            "doi": doi,
            "url": url,
            "citation": citation
        })

    return papers


def extract_json_from_response(text):  ### cette fonction rend le programme plus robuste face aux petites erreurs de format du modèle.
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"No valid JSON found in model response: {text[:300]}")


def screen_paper(paper, timeout_seconds=TIMEOUT_SECONDS):    
### Le mode /no_think est activé afin de garantir une sortie structurée et directement 
### exploitable en format JSON, sans texte supplémentaire. Cela permet d’assurer 
### la stabilité du processus automatisé de filtrage des articles et d’éviter les erreurs de parsing
    
    prompt = f"""
    /no_think
You are a strict academic screening assistant.

Base your decision ONLY on the provided title, abstract, and keywords.
Do not use outside knowledge.
Do not invent information.
If evidence is insufficient, choose uncertain.

Return ONLY a JSON object.
Do not include explanations outside JSON.
Do not use markdown.

The JSON must have exactly these fields:
{{
  "include": "include" or "exclude" or "uncertain",
  "main_reason": "short reason based only on the abstract",
  "matched_inclusion_criteria": [],
  "matched_exclusion_criteria": [],
  "paper_type": "original research" or "review/survey" or "unclear",
  "modalities_detected": [],
  "task_detected": "",
  "confidence": "high" or "medium" or "low"
}}

INCLUSION CRITERIA:
{INCLUSION_CRITERIA}

EXCLUSION CRITERIA:
{EXCLUSION_CRITERIA}

Title:
{paper["title"]}

Authors:
{paper["authors"]}

Abstract:
{paper["abstract"]}

Keywords:
{paper["keywords"]}
"""

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False,
        "options": {
            "temperature": 0,        ### rend les réponses plus stables et moins créatives
            "num_predict": 1200      ### Le nombre maximal de tokens que vous êtes autorisé à générer dans votre réponse. Je l’ai déterminé empiriquement pour notre travail.
        }
    }

    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=timeout_seconds
    )

    response.raise_for_status()
    result = response.json()

    raw_text = result["message"]["content"]
    print("RAW MODEL RESPONSE:", raw_text[:500])

    return extract_json_from_response(raw_text)


def make_error_decision(error_message):
    return {
        "include": "uncertain",
        "confidence": "low",
        "main_reason": f"Model, timeout, or JSON error: {error_message}",
        "paper_type": "unclear",
        "modalities_detected": [],
        "task_detected": "",
        "matched_inclusion_criteria": [],
        "matched_exclusion_criteria": []
    }


def save_checkpoint(records):
    df_temp = pd.DataFrame(records)
    df_temp.to_csv(CHECKPOINT_CSV, index=False)
    df_temp.to_excel(CHECKPOINT_EXCEL, index=False)


if Path(CHECKPOINT_CSV).exists():               ########## Avant de commencer, le code vérifie si un checkpoint existe.

                                                ########### S’il existe, il recharge les articles déjà traités et évite de les refaire
    existing_df = pd.read_csv(CHECKPOINT_CSV)
    all_papers = existing_df.to_dict("records")
    processed_urls = set(existing_df["url"].dropna().astype(str))
    print(f"Resuming from checkpoint: {len(all_papers)} papers already processed.")
else:
    all_papers = []
    processed_urls = set()
    print("No checkpoint found. Starting from zero.")


for file in FILES:
    text = Path(file).read_text(encoding="utf-8", errors="ignore")

    print("\n==============================")
    print(f"Reading file: {file}")
    print("URLs in raw file:", len(re.findall(r"URL:", text, flags=re.IGNORECASE)))

    papers = split_ieee_txt(text)

    print(f"{file}: Found {len(papers)} papers")

    for i, paper in enumerate(papers, start=1):

        if paper["url"] in processed_urls:
            print(f"Skipping already processed paper {i}/{len(papers)}")
            continue

        print(f"\nProcessing {i}/{len(papers)}: {paper['title'][:100]}")

        try:
            decision = screen_paper(paper)

        except Exception as e:
            decision = make_error_decision(str(e))
            print(f"Problem with this paper. Marked uncertain. Error: {e}")

        record = {
            "file": file,
            "title": paper["title"],
            "authors": paper["authors"],
            "doi": paper["doi"],
            "url": paper["url"],
            "abstract": paper["abstract"],
            "keywords": paper["keywords"],
            "decision": decision.get("include", "uncertain"),
            "confidence": decision.get("confidence", "low"),
            "main_reason": decision.get("main_reason", ""),
            "paper_type": decision.get("paper_type", "unclear"),
            "modalities_detected": "; ".join(decision.get("modalities_detected", [])),
            "task_detected": decision.get("task_detected", ""),
            "matched_inclusion_criteria": "; ".join(decision.get("matched_inclusion_criteria", [])),
            "matched_exclusion_criteria": "; ".join(decision.get("matched_exclusion_criteria", []))
        }

        all_papers.append(record)
        processed_urls.add(paper["url"])

        save_checkpoint(all_papers)

        print(f"Decision: {record['decision']} | Confidence: {record['confidence']}")
        print(f"Checkpoint saved after {len(all_papers)} total papers.")


df = pd.DataFrame(all_papers)

df.to_excel(OUTPUT_EXCEL, index=False)
df.to_csv(OUTPUT_CSV, index=False)

print("\n==============================")
print(f"Done. Screened {len(df)} papers.")
print(f"Saved final Excel: {OUTPUT_EXCEL}")
print(f"Saved final CSV: {OUTPUT_CSV}")
print(f"Saved checkpoint Excel: {CHECKPOINT_EXCEL}")
print(f"Saved checkpoint CSV: {CHECKPOINT_CSV}")

end = time.perf_counter()
print(f"Execution time: {end - start:.2f} seconds")

No checkpoint found. Starting from zero.

Reading file: ieee_results_1.txt
URLs in raw file: 100
Detected URLs: 100
Detected paper chunks: 100
ieee_results_1.txt: Found 100 papers

Processing 1/100: Estimating Perceptual Attributes of Haptic Textures Using Visuo-Tactile Data,
RAW MODEL RESPONSE: {
  "include": "include",
  "main_reason": "The paper combines visual and tactile modalities for texture recognition, a perception-level task.",
  "matched_inclusion_criteria": [
    "texture recognition",
    "multimodal or cross-modal representation learning"
  ],
  "matched_exclusion_criteria": [],
  "paper_type": "original research",
  "modalities_detected": [
    "vision / visual",
    "tactile / haptic"
  ],
  "task_detected": "texture recognition",
  "confidence": "high"
}
Decision: include | Confidence: high
Checkpoint saved after 1 total papers.

Processing 2/100: Classification Method of Visual-Tactile Fusion Dataset Based on CNN-TCN,
RAW MODEL RESPONSE: {
  "include": "include",
  "m